In [1]:
import sympy as sp
import numpy as np
from SumOfSquares import SOSProblem, poly_opt_prob

In [2]:
# Defines symbolic variables and polynomial
x, y = sp.symbols('x y')
p = 2*x**4 + 2*x**3*y - x**2*y**2 + 5*y**4
prob = SOSProblem()

# Adds Sum-of-Squares constaint and solves problem
const = prob.add_sos_constraint(p, [x, y])
prob.solve(solver='mosek')

# Prints Sum-of-Squares decomposition
print(sum(const.get_sos_decomp()))

1.047*x**4 + 5.0*(-0.306*x**2 + y**2)**2 + 2.057*(0.486*x**2 + x*y)**2


In [3]:
x, y = sp.symbols('x y')
p = x**4*y**2 + x**2*y**4 - 3*x**2*y**2 + 1
prob = SOSProblem()
prob.add_sos_constraint(p, [x, y])
prob.solve(solver='mosek') # Raises SolutionFailure error due to infeasibility

SolutionFailure: Code 3: Primal solution state claimed infeasible but optimality is required (primals=True).

In [ ]:
x, y, s, t = sp.symbols('x y s t')
p = s*x**6 + t*y**6 - x**4*y**2 - x**2*y**4 - x**4 \
    + 3*x**2*y**2 - y**4 - x**2 - y**2 + 1
prob = SOSProblem()
prob.add_sos_constraint(p, [x, y])
sv, tv = prob.sym_to_var(s), prob.sym_to_var(t)
prob.solve(solver='mosek')
prob.set_objective('min', sv+tv)
print(sv.value, tv.value)

In [ ]:
x, y, t = sp.symbols('x y t')
p1 = t*(1 + x*y)**2 - x*y + (1 - y)**2
p2 = (1 - x*y)**2 + x*y + t*(1 + y)**2
prob = SOSProblem()
prob.add_sos_constraint(p1, [x, y])
prob.add_sos_constraint(p2, [x, y])
tv = prob.sym_to_var(t)
prob.set_objective('min', tv)
prob.solve()
print(tv.value)
# returns t ~ 0.25

In [ ]:
x, y, t = sp.symbols('x y t')
p = x**4 + x**2 - 3*x**2*y**2 + y**6
prob = SOSProblem()
# Use Newton polytope reduction
prob.add_sos_constraint(p-t, [x, y],sparse=True)
tv = prob.sym_to_var(t)
prob.set_objective('max', tv)
prob.solve(solver='mosek')
print(prob.value)
# Returns the lower bound -.177979

In [4]:
x, y, t = sp.symbols('x y t')
p = 4*x**2 -21/10* x**4 +1/3*x**6 + x*y - 4*y**2 + 4*y**4
prob = SOSProblem()
# Use Newton polytope reduction
prob.add_sos_constraint(p-t, [x, y],sparse=False)
tv = prob.sym_to_var(t)
prob.set_objective('max', tv)
prob.solve(solver='mosek')
print(prob.value)
# Returns the lower bound -.177979

-1.031628453299999


In [18]:
state = sp.Matrix(sp.symbols('x y', real=True))
x = state[0,0]
y = state[1,0]
coeffs = []
V = sp.zeros(1,1)
for i in range(5) :
    for j in range(5) :
        if i + j > 4 : 
            continue
        coeffs.append(sp.symbols('c' + str(i) + str(j)))
        V[0,0] += coeffs[-1] * x**i * y**j
Vx = V.jacobian(state)
f = sp.zeros(2,1)
f[0,0] = -y + 3/2*x**2 -1/2*x**3
f[1,0] = 3*x-y
prob = SOSProblem()
cv = []
for c in coeffs :
    cv.append(prob.sym_to_var(c))
p1 = prob.add_sos_constraint(V[0,0], [x,y])
p2 = prob.add_sos_constraint(-(Vx@f)[0,0], [x,y])
# prob.set_objective('min', 1)
prob.solve(solver='mosek')



<primal feasible solution pair (claimed optimal) from mosek>

Matrix([
[           0.916*(0.058*x**2 - 0.035*x*y - 0.044*y**2 + 1)**2],
[0.872*(-0.168*x**2 + 0.041*x*y - 0.285*x - 0.174*y**2 + y)**2],
[           1.811*(0.011*x**2 - 0.266*x*y + x - 0.006*y**2)**2],
[                    0.021*(-x**2 - 0.016*x*y + 0.907*y**2)**2],
[                                  0.12*(-0.115*x**2 + x*y)**2],
[                                                    0.18*x**4]])